<a href="https://colab.research.google.com/github/frazeui/Fraud-Detection-AI-Agent/blob/main/fraud_detection_agent_v3_multiagent_with_multimodal_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
!pip install fastapi uvicorn nest_asyncio pyngrok pydantic peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
!pip install langchain-groq langgraph-checkpoint-sqlite
!pip install langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 9.4 MB/s eta 0:00:00


In [4]:
import os
from pydantic import BaseModel,Field
from typing import Annotated ,TypedDict,Literal
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command,interrupt
import sqlite3
from tenacity import retry,wait_random_exponential,stop_after_attempt
from pydantic import Field,BaseModel
from typing import Optional,Literal
from datetime import date

In [61]:
class Risk_Assessment(BaseModel):

    overall_risk:Literal["LOW","MEDIUM","HIGH"]=Field(description="Overal risk detection will be on the Risk Analyst's findings")

    recommendation:Literal["APPROVE","REVIEW","BLOCK"]=Field(description="Action to take for LOW:APPROVE,REVIEW for MEDIUM and BLOCK for HIGH ")

    justification:str=Field(description="Clear reasoning for the specific findings from Risk Analyst")

In [62]:
# EK HI CELL — environment + agent setup dono
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("langsmith")
os.environ["LANGCHAIN_PROJECT"] = "fraud-detection-MultiAgent"


In [63]:
user_profiles={
    "user_101":{"home_country":"UAE","avg_transactions":500},
    "user_102":{"home_country":"Pakistan","avg_transactions":200},
}

# **Agent Functions**

In [64]:
import time
@tool
def check_amount_risk(amount:float,user_id:str)->str:
    """verify that either the amount which the users is withdrawing is it average based or fraud """

    time.sleep(2)
    profile=user_profiles.get(user_id)

    print(f"[Debug] finished at {time.time():.2f}")

    if not profile: return "Error: User profile not found "

    avg=profile["avg_transactions"]

    if amount>avg*10:
        return f"High Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    elif amount>avg*3:
        return f"Medium Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    else:
        return f"Low risk: [{amount}] is wiithin the normal range"

In [65]:
@tool
def check_velocity(transactions_count_last_hour:int)->str:
    """verify the number of transactions last hour either it's a fraud or not """
    time.sleep(2)

    print(f"[Debug] finished at {time.time():.2f}")
    if transactions_count_last_hour>=5:
        return f"High Risk: {transactions_count_last_hour} times which is unusual velocity"

    elif transactions_count_last_hour>=3 and transactions_count_last_hour<5:
        return f"Medium Risk: {transactions_count_last_hour} time which is unusual velocity"

    else:
        return f"Low Risk: {transactions_count_last_hour} times which is normal range"

In [66]:
@tool
def check_location_mismatch(user_id:str,transactions_country:str)->str:
    """Tell about the country that either it's matched or not"""

    time.sleep(2)

    profile=user_profiles.get(user_id)
    if not profile:return "User profile not found"

    home=profile["home_country"]

    print(f"[Debug] finished at {time.time():.2f}")

    if home.lower()!=transactions_country.lower():
        return f"Medium Risk: Transactoin country is [{transactions_country}] and Home country is [{home}]"
    else:
        return f"Low Risk: Transaction country [{transactions_country}] and Home country [{home}] matches "

In [67]:
risk_tools=[check_amount_risk,check_velocity,check_location_mismatch]

In [68]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]
    document_image:Optional[str]

In [69]:
from google.colab import userdata

groq_api = userdata.get("groq_api")

# Text/tool-calling model — risk analysis + decision agent
text_llm = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api)
risk_analyst_llm = text_llm.bind_tools(risk_tools)
decision_llm = text_llm
structured_decision_llm = decision_llm.with_structured_output(Risk_Assessment, method="json_mode")

# Vision model — document verification only
vision_llm = ChatGroq(model="qwen/qwen3.6-27b", max_tokens=1500, api_key=groq_api)
document_verification_llm = vision_llm.with_structured_output(Document_Extraction_Result)

In [70]:
from groq import Groq

client = Groq(api_key=groq_api)

print("Available Groq Models:")
for model in client.models.list().data:
    print(f"- {model.id}")

Available Groq Models:
- groq/compound-mini
- openai/gpt-oss-20b
- openai/gpt-oss-120b
- canopylabs/orpheus-arabic-saudi
- meta-llama/llama-prompt-guard-2-22m
- meta-llama/llama-prompt-guard-2-86m
- whisper-large-v3
- openai/gpt-oss-safeguard-20b
- groq/compound
- canopylabs/orpheus-v1-english
- whisper-large-v3-turbo
- qwen/qwen3.6-27b
- allam-2-7b


In [108]:
RISK_ANALYST_PROMPT="""
You are a Risk Analyst Agent. Your Only Work is to run risk checks on a transactions using tools - you don't make final decisions or recommendations.


Rules:
i-Always run THREE Tools:[check_amount_risk,check_velocity,check_location_mismatch]
ii-First check the one category , then other and finish in sequential way.
'Example1: suppose,first you observe amount_risk which was 200 after getting you fed up the result in your mind and then you observed velocity of it you also fed up in your mind and then you observed location which you also fed up then you checked it up with the previous data if it's match then either it's related or not.'
iii-After getting all results,summarize each one with factual answer-quote exact tool output don't hallucinate in it
'like: If the overall Risk is LOW, print=>APPROVE'
'like:If the overall Risk is MEDIUM, print=>REVIEW'
'like:If the overall Risk is HIGH, print=>BLOCK'
iv-Don't give the final clarification like(LOW,MEDIUM,HIGH) or (BLOCK/REVIEW/APPROVE),that's the job of Decision Agent not yours
v-End your summary clearly and send next to another agent which can read your summary easily """
DECISION_AGENT_PROMPT = """
You are Decision Agent. Your job is to review the Risk Analyst's findings and respond with a JSON object with exactly these three fields: overall_risk, recommendation, justification.

overall_risk must be one of: LOW, MEDIUM, HIGH
recommendation must be one of: APPROVE, REVIEW, BLOCK

Classification Rules - apply these to the actual findings given to you, do not use a default value:
- HIGH: at least ONE signal (amount, velocity, or location) is independently HIGH RISK, OR two-or-more signals are MEDIUM RISK.
- MEDIUM: exactly ONE signal is MEDIUM RISK and no signal is HIGH RISK.
- LOW: ALL signals are LOW RISK.

EXAMPLE 1:
Findings: "Amount: HIGH RISK (20x average). Velocity: LOW RISK. Location: LOW RISK."
Output: overall_risk=HIGH, recommendation=BLOCK

EXAMPLE 2:
Findings: "Amount: MEDIUM RISK. Location: MEDIUM RISK. Velocity: LOW RISK."
Output: overall_risk=HIGH, recommendation=BLOCK (two MEDIUM signals escalate to HIGH)

EXAMPLE 3:
Findings: "Amount: LOW RISK. Velocity: LOW RISK. Location: LOW RISK."
Output: overall_risk=LOW, recommendation=APPROVE

Recommendation mapping: LOW->APPROVE, MEDIUM->REVIEW, HIGH->BLOCK

Don't hallucinate or guess from yourself about it just decide on the basis of the 'RISK_ANALYST' tool and give the results

Additional Rules:
1. Justification must reference the specific findings from the Risk Analyst - never invent new data.
2. When referencing the amount finding, use the exact multiplier format (e.g., '4.0x higher'), never percentage.
3. Use exactly these field names: overall_risk, recommendation, justification.
"""

# **Image Verification**

In [109]:
class Document_Extraction_Result(BaseModel):
    document_type:str=Field(description="Type of document e.g.,Passport,Invoice,bank statement")
    name:Optional[str]=Field(default=None,description="Full name found on the document,if visible")
    id_number:Optional[str]=Field(default=None,description="ID/Document number ,if visible")
    date_of_birth:Optional[str]=Field(default=None,description="Date of Expiry,if visible")


    appears_authentic:Literal["yes","no"]=Field(description="Overall authenticity assessment")
    red_flags:list[str]=Field(description="List of specific visuals if any,Empy if list is none")
    font_consistency:Literal["consistent","inconsistent","cannot_determine"]=Field(description="Tells about the fonts/styles throughout the document are consistent or not ")
    tampering_indicators:list[str]=Field(description="Tells about wether the image is editted or tempered or any kind of changes")
    confidence_level:Literal["Low","Medium","High"]=Field(description="Overall confidence level of the document")

In [110]:
import base64
def encode_image_to_base64(image_path:str)->str:
    with open(image_path,"rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


In [111]:
def extract_document_fields(base64_image:str)->Document_Extraction_Result:


    prompt="""Carefully examine this document for signs of authenticity or tampering.
    Check for: font consistency, unusual placeholder text, digital editing artifacts,misaligned elements, or anything that suggests this is not a genuine document.Be specific about what you observe."""

    message=HumanMessage(content=[
        {"type":"text",
        "text":prompt},
         {"type":"image_url",
        "image_url":{"url":f"data:image/jpeg;base64,{base64_image}"}}
    ])

    result:Document_Extraction_Result=document_verification_llm.invoke([message])

    return result

# **Nodes**

In [112]:
def document_verification_node(state: AgentState):
    base64_image = state.get("document_image")

    if not base64_image:
        return {"messages": [AIMessage(content="No document provided for verification - skipping document check.")]}

    try:
        extraction = extract_document_fields(base64_image=base64_image)
        summary_text = (
            f"[Document Verification]\n"
            f"Document Type: {extraction.document_type}\n"
            f"Appears Authentic: {extraction.appears_authentic}\n"
            f"Red Flags: {extraction.red_flags}\n"
            f"Confidence Level: {extraction.confidence_level}"
        )
    except Exception as e:
        print(f"[Document Verification Error] {type(e).__name__}: {e}")
        # Graceful fallback - crash mat karo, flag karo ke manual review chahiye
        summary_text = (
            f"[Document Verification]\n"
            f"Automated document verification failed due to a technical issue. "
            f"This transaction requires MANUAL document review before approval."
        )

    return {"messages": [AIMessage(content=summary_text)]}

In [113]:
@retry(wait=wait_random_exponential(min=1,max=20),stop=stop_after_attempt(5))
def risk_analyst_node(state:AgentState):

    messages=state["messages"]
    if not any (isinstance(m,SystemMessage) for m in messages):
        messages=[SystemMessage(content=RISK_ANALYST_PROMPT)]+messages
    response=risk_analyst_llm.invoke(messages)
    return {"messages":[response]}
risk_tool_node=ToolNode(risk_tools)

In [114]:
@retry(wait=wait_random_exponential(min=1,max=20),stop=stop_after_attempt(3))
def decision_agent_node(state:AgentState):
    risk_findings=None
    for m in reversed(state["messages"]):
        if isinstance(m,AIMessage) and not m.tool_calls and m.content:
            risk_findings=m.content
            break
    decision_input=[
        SystemMessage(content=DECISION_AGENT_PROMPT),
        HumanMessage(content=f"Risk Analyst's findings:\n\n{risk_findings}\n\nProvide your final risk classification and recommendation.")
    ]
    structure_result:Risk_Assessment=structured_decision_llm.invoke(decision_input)
    formatted_text=(
        f"Overall Risk: {structure_result.overall_risk}\n"
        f"Recommendation: {structure_result.recommendation}\n"
        f"Justification: {structure_result.justification}"
    )
    response=AIMessage(content=formatted_text)
    return {"messages":[response],"structure_assessment":structure_result}


In [115]:
def should_continue_risk_analysis(state:AgentState):
    last_message=state["messages"][-1]
    if getattr(last_message,"tool_calls",None):
        return "risk_tools"
    return "decision_agent"

In [116]:
def human_review_node(state:AgentState):
    print("Human Review node reached")
    last_message=state["messages"][-1]
    assessment_text=last_message.content

    needs_review="HIGH" in assessment_text.upper() or 'BLOCK' in assessment_text.upper()

    if not needs_review:return {"messages":[]}
    human_decision = interrupt({
        "question": "Decision Agent flagged this as HIGH risk / BLOCK. Please confirm.",
        "decision_agent_assessment": assessment_text
    })
    confirmation_message=HumanMessage(content=f"[Human Review]: {human_decision}")
    return {"messages":[confirmation_message]}

In [117]:
graph=StateGraph(AgentState)
graph.add_node("risk_analyst",risk_analyst_node)
graph.add_node("risk_tools",risk_tool_node)
graph.add_node("document_verification",document_verification_node)
graph.add_node("decision_agent",decision_agent_node)
graph.add_node("human_review",human_review_node)

graph.set_entry_point("risk_analyst")

graph.add_conditional_edges("risk_analyst",should_continue_risk_analysis,{"risk_tools":"risk_tools","decision_agent":"document_verification",END:END})

graph.add_edge("risk_tools","risk_analyst")
graph.add_edge("document_verification","decision_agent")
graph.add_edge("decision_agent","human_review")
graph.add_edge("human_review",END)

conn=sqlite3.connect("multi_agent_fraud_memory.db",check_same_thread=False)
memory=SqliteSaver(conn)

app=graph.compile(checkpointer=memory)

In [118]:
print(list(app.get_graph().nodes.keys()))

['__start__', 'risk_analyst', 'risk_tools', 'document_verification', 'decision_agent', 'human_review', '__end__']


In [119]:
from fastapi import FastAPI,Form,UploadFile,File
from pydantic import BaseModel
from typing import Annotated
import time
import base64

api=FastAPI(title="Fraud Detection Agent")


class HumanDecisionRequest(BaseModel):
    thread_id:str
    decision:str

@api.post("/analyze_transactions_with_documents")
async def analyze_transactions_with_documents(thread_id:str=Form(...),description:str=Form(...),document: UploadFile =File(...)):
    # try:
        start=time.time()
        image_bytes=await document.read()
        base64_image=base64.b64encode(image_bytes).decode('utf-8')

        config={"configurable":{"thread_id":thread_id}}
        result = app.invoke({
            "messages": [
                HumanMessage(content=description)
            ],
            "document_image":base64_image
        },config=config)


        if "__interrupt__" in result:
            interrupt_data=result["__interrupt__"][0].value
            return {
                "status":"PENDING REQUEST",
                "thread_id":thread_id,
                "ai_assessment":interrupt_data["decision_agent_assessment"],
                "message":"High risk detected .Call/human decision with your decision",
                "processing_time_seconds":round(time.time()-start,2)

            }
        return{
            "status": "COMPLETED",
            "thread_id": thread_id,
            "final_result": result["messages"][-1].content,
                "processing_time_seconds":round(time.time()-start,2)
        }
    # except Exception as e:
    #     return {
    #         "status":"Error",
    #         "error":str(e)
    #     }

@api.post("/human_decision")
def human_decision(req:HumanDecisionRequest):

    config={"configurable":{"thread_id":req.thread_id}}

    result=app.invoke(Command(resume=req.decision),config=config)

    return {
        "status":"COMPLETED",
        "thread_id":req.thread_id,
        "final_result":result["messages"][-1].content
    }


@api.get("/")
def health_check():
    return {"status":"Fraud Detection AI agent API is running...."}

print(f"FastAPI app ready")

FastAPI app ready


In [120]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from google.colab import userdata

nest_asyncio.apply()

ngrok_token=userdata.get("ngrok_token")
ngrok.set_auth_token(ngrok_token)

public_url=ngrok.connect(8000)

print(f"Public URL: {public_url}")
print(f"API docs: {public_url}/docs")

config=uvicorn.Config(api,host="0.0.0.0",port=8000,log_level="info")
server=uvicorn.Server(config)
await server.serve()


Public URL: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"
API docs: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [1108]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1108]


# **Agent Evaluation**

In [121]:
import re

TEST_CASES = [
    {
        "id": "TC1_low_normal",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=UAE, transaction_count_last_hour=1",
        "expected_risk": "LOW",
    },
    {
        "id": "TC2_high_amount_only",
        "description": "Analyze this transaction: user_id=user_101, amount=15000, transaction_country=UAE, transaction_count_last_hour=1",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC3_high_velocity_only",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=UAE, transaction_count_last_hour=7",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC4_location_mismatch_only",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=Nigeria, transaction_count_last_hour=1",
        "expected_risk": "MEDIUM",
    },
    {
        "id": "TC5_all_high_signals",
        "description": "Analyze this transaction: user_id=user_101, amount=20000, transaction_country=Nigeria, transaction_count_last_hour=8",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC6_borderline_amount",
        "description": "Analyze this transaction: user_id=user_101, amount=1800, transaction_country=UAE, transaction_count_last_hour=2",
        "expected_risk": "MEDIUM",
    },
    {
        "id": "TC7_normal_user2",
        "description": "Analyze this transaction: user_id=user_102, amount=150, transaction_country=Pakistan, transaction_count_last_hour=1",
        "expected_risk": "LOW",
    },
    {
        "id": "TC8_user2_high_amount",
        "description": "Analyze this transaction: user_id=user_102, amount=3000, transaction_country=Pakistan, transaction_count_last_hour=1",
        "expected_risk": "HIGH",
    },
]


In [122]:
def extract_risk(text:str)->str:
    text=text.upper()
    if re.search(r'\bHIGH\b',text):
        return 'HIGH'
    elif re.search(r'\bMEDIUM\b',text):
        return 'MEDIUM'
    elif re.search(r'\bLOW\b',text):
        return 'LOW'
    return 'UNKOWN'

In [123]:
def run_singal_eval(test_case:dict,thread_id:str)->dict:
    config={"configurable":{"thread_id":thread_id}}

    result=app.invoke({
        "messages":HumanMessage(content=test_case["description"])},config=config)

    if '__interrupt__' in result:
        assessment_text=result['__interrupt__'][0].value["decision_agent_assessment"]

    else:
        assessment_text=result["messages"][-1].content

    predicted_risk=extract_risk(assessment_text)

    return {
        "id":test_case["id"],
        "expected_risk":test_case["expected_risk"],
        "predicted":predicted_risk,
        "correct":predicted_risk==test_case["expected_risk"],
        "raw_response":assessment_text
    }

In [124]:
def run_evaluation():
    results=[]

    for i,test_case in enumerate(TEST_CASES):

        thread_id=f"eval-{test_case['id']}-{i}"
        print(f"Running {test_case['id']}...",end=" ")

        try:
            eval_results=run_singal_eval(test_case,thread_id)
            results.append(eval_results)
            status=("PASS" if eval_results["correct"] else 'FAIL')
            print(f"{status} (expected: {eval_results['expected_risk']},got:{eval_results['predicted']})\n")

        except Exception as e:
            print(f"Error: {e}")
            results.append({
                "id":test_case["id"],
                "expected_risk":test_case['expected_risk'],
                "predicted":'ERROR',
                "correct":False,
                "raw_response":str(e)
                })

    return results

In [125]:
def calculate_results(results:list):
    total=len(results)
    correct=sum(1 for r in results if r["correct"])
    accuracy=(correct/total)*100 if total>0 else 0

    false_negative=[
        r for r in results
        if r["expected_risk"] in ('HIGH','MEDIUM') and r["predicted"] == ('LOW',)
    ]

    false_positives=[
        r for r in results
        if r["expected_risk"] in ('LOW',) and r["predicted"] == ('HIGH','MEDIUM',)
    ]

    print(f"\n{'='*60}")
    print('EVALUATION REPORT')
    print(f"\n{'='*60}")
    print(f"Total Cases: {total}")
    print(f"Correct Cases: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"False negatives: {len(false_negative)}")
    for fn in false_negative:
        print(f"False negative id {fn['id']} expected: {fn['expected_risk']} but got: {fn['predicted']}")

    print(f"False positives: {len(false_positives)}")
    for fn in false_positives:
        print(f"False positive id {fn['id']} expected: {fn['expected_risk']} but got: {fn['predicted']}")

    print(f"\n{'='*60}")

    return {
        "accuracy":accuracy,
        "false_negatives":false_negative,
        "false_positives":false_positives,
        "total":total
    }



In [ ]:
if __name__=='__main__':
    results=run_evaluation()
    metrics=calculate_results(results)